In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from matplotlib import pyplot as plt
from jordanutils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


rng = np.random.default_rng(seed=111)

In [15]:
def train_and_test_model(train_mode, test_mode, schur, verbose=1):
    d = 5
    dataset_size = 50000
    manager = LabelsManager([0], [1], [2], [3], [4], dataset_size=dataset_size)
    X, y = generate_testset(d, manager, mode=train_mode, schur=schur)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    X_train = X_train.astype(np.float32)
    y_train = np.array(y_train, dtype=np.int32)
    y_val = np.array(y_val, dtype=np.int32)

    model1 = tf.keras.Sequential(
        [
            tf.keras.layers.Flatten(input_shape=(d, d)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(d),
        ]
    )

    model1.compile(
        optimizer="adam",
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )

    model1.fit(
        X_train,
        y_train,
        epochs=50,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=verbose,
    )

    manager.dataset_size = 1000
    X_test, y_test = generate_testset(d, manager, mode=test_mode, schur=schur)

    y_predicted = model1.predict(X_test)
    y_predicted = np.argmax(y_predicted, axis=1)

    return y_test, y_predicted

In [16]:
y_test, y_pred = train_and_test_model("random", "random", schur=False, verbose=1)
print(classification_report(y_test, y_pred))

Epoch 1/50


c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6250/6250 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.5127 - loss: 1.0435 - val_accuracy: 0.5861 - val_loss: 0.8541
Epoch 2/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.6034 - loss: 0.8150 - val_accuracy: 0.6157 - val_loss: 0.7844
Epoch 3/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.6291 - loss: 0.7656 - val_accuracy: 0.6392 - val_loss: 0.7509
Epoch 4/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.6433 - loss: 0.7387 - val_accuracy: 0.6445 - val_loss: 0.7346
Epoch 5/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.6542 - loss: 0.7179 - val_accuracy: 0.6591 - val_loss: 0.7074
Epoch 6/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.6638 - loss: 0.6954 - val_accuracy: 0.6678 - val_loss: 0.6944
Epoch 7/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.6733 - loss: 0.6821 - val_accuracy: 0.6739 - val_loss: 0.6804
Epoch 8/50
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.6788 - loss: 0.6688 - val

In [17]:
y_test, y_pred = train_and_test_model("random", "random", schur=True, verbose=0)
print(classification_report(y_test, y_pred))

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.81      0.88      0.84      1000
           2       0.57      0.59      0.58      1000
           3       0.44      0.38      0.41      1000
           4       0.58      0.59      0.58      1000

    accuracy                           0.69      5000
   macro avg       0.68      0.69      0.68      5000
weighted avg       0.68      0.69      0.68      5000

